# Kapitola 3: Přiřazování rolí (Role Prompting)

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku nastavení pro načtení vašeho API klíče a vytvoření pomocné funkce `get_completion`.

In [ ]:
```python
# Import vestavěnou knihovnu regulárních výrazů v Pythonu
import re
import boto3
import json

# Import modulu hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Načtení proměnné MODEL_NAME z úložiště IPython
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(prompt,system=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')
```

---

## Lekce

Pokračujeme v tématu, že Claude nemá žádný kontext kromě toho, co řeknete, je někdy důležité **promptovat Clauda, aby přijal specifickou roli (včetně veškerého potřebného kontextu)**. Toto je také známé jako role prompting. Čím více detailů k roli, tím lépe.

**Připravení Clauda s rolí může zlepšit jeho výkon** v různých oblastech, od psaní po programování až po shrnování. Je to jako když lidem někdy pomůže, když jim řeknete, aby "mysleli jako ______". Role prompting může také změnit styl, tón a způsob Claudeovy odpovědi.

**Poznámka:** Role prompting může probíhat buď v system promptu, nebo jako součást uživatelského sdělení.

### Příklady

V níže uvedeném příkladu vidíme, že bez role promptingu poskytuje Claude **přímou a nestylizovanou odpověď**, když je požádán o jednovětnou perspektivu na skateboarding.

Když však připravíme Claudea, aby přijal roli kočky, Claudeova perspektiva se změní, a tím se **Claudeův tón odpovědi, styl a obsah přizpůsobí nové roli**.

**Poznámka:** Bonusová technika, kterou můžete použít, je **poskytnout Claudovi kontext o jeho zamýšleném publiku**. Níže jsme mohli upravit prompt tak, aby Claudovi také řekl, s kým by měl mluvit. "Jsi kočka" vyvolá zcela jinou odpověď než "jsi kočka mluvící k davu skateboardistů".

Zde je prompt bez role promptingu v systémovém promptu:

In [ ]:
```python
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

Zde je stejná uživatelská otázka, ale s použitím role promptu.

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

Můžete použít role prompting jako způsob, jak přimět Claude, aby napodoboval určité styly psaní, mluvil určitým hlasem nebo řídil složitost svých odpovědí. **Role prompting může také zlepšit schopnost Claude vykonávat matematické nebo logické úkoly.**

Například v níže uvedeném příkladu existuje jednoznačně správná odpověď, kterou je ano. Nicméně Claude se mýlí a myslí si, že mu chybí informace, což není pravda:

In [ ]:
```python
# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatý člověk na neženatého člověka?"

# Vytisknout Claudeovu odpověď
print(get_completion(PROMPT))
```

Nyní, co když **nastavíme Claudea, aby fungoval jako logický bot**? Jak to změní Claudeovu odpověď?

Ukazuje se, že s tímto novým přiřazením role to Claude zvládne správně. (I když ne nutně ze všech správných důvodů)

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatý člověk na neženatého člověka?"

# Vytisknout odpověď Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

**Poznámka:** Co se během tohoto kurzu naučíte, je, že existuje **mnoho technik prompt engineeringu, které můžete použít k dosažení podobných výsledků**. Které techniky použijete, záleží na vás a vašich preferencích! Doporučujeme vám **experimentovat a najít si svůj vlastní styl prompt engineeringu**.

Pokud byste chtěli experimentovat s prompty z lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

## Cvičení
- [Cvičení 3.1 - Oprava matematiky](#exercise-31---math-correction)

### Cvičení 3.1 - Oprava matematiky
V některých případech **může mít Claude problémy s matematikou**, dokonce i s jednoduchou matematikou. Níže Claude nesprávně vyhodnotí matematický problém jako správně vyřešený, i když je ve druhém kroku zjevná aritmetická chyba. Všimněte si, že Claude chybu skutečně zachytí při postupném procházení, ale nedospěje k závěru, že celkové řešení je špatné.

Upravte `PROMPT` a / nebo `SYSTEM_PROMPT`, aby Claude ohodnotil řešení jako `nesprávně` vyřešené, místo správně vyřešeného.

In [ ]:
```python
# Systemový prompt - pokud nechcete použít systémový prompt, můžete tuto proměnnou nechat nastavenou na prázdný řetězec
SYSTEM_PROMPT = ""

# Prompt
PROMPT = """Is this equation solved correctly below?

2x - 3 = 9
2x = 6
x = 3"""

# Získání odpovědi od Claude
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    if "incorrect" in text or "not correct" in text.lower():
        return True
    else:
        return False

# Tisk odpovědi od Claude a odpovídající hodnocení
print(response)
print("\n--------------------------- HODNOCENÍ ---------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_3_1_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Přejeme šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT))
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Vytiskni Claudeovu odpověď
print(get_completion(PROMPT, SYSTEM_PROMPT))
```

In [ ]:
```python
# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatý člověk na neženatého člověka?"

# Vytiskni Claudeovu odpověď
print(get_completion(PROMPT))
```

In [ ]:
```python
# Systémový prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack se dívá na Anne. Anne se dívá na George. Jack je ženatý, George není, a nevíme, zda je Anne vdaná. Dívá se ženatý člověk na neženatého člověka?"

# Vytisknout odpověď od Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))
```